In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("API_KEY")

In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key = groq_key,
)

response = llm.invoke("The first person to land on moon was...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon's surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That's one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.


In [11]:
from langchain_community.document_loaders import WebBaseLoader

#loader = WebBaseLoader("https://jobs.nike.com/job/R-33460")
loader = WebBaseLoader("https://careers.nike.com/fr/retail-associate-pt-nike-staten-island-14-29-hours-week/job/R-79939")
page_data = loader.load().pop().page_content
print(page_data)





















Retail Associate, PT - Nike Staten Island (14-29 hours/week)












































Passer au contenu principal
Ouvrir l'assistant virtuel









Accueil
Domaines d'activité
Total Rewards
Life@Nike
Objectif









Langue





Sélectionnez une langue

Deutsch
English
Español
                                (España)
Español
                                (América Latina)
Français
Italiano
Nederlands
Polski
Tiếng Việt
Türkçe
简体中文
繁體中文
עִברִית
한국어
日本語








Carrières


















Fermer le menu







Carrières






Discussion




Accueil
Domaines d'activité
Total Rewards
Life@Nike
Objectif









Jordan Careers







Converse Careers










Langue













Menu





Revenir au menu précédent



Sélectionnez une langue

Deutsch
English
Español
                                (España)
Español
                                (América Latina)
Français
Italiano
Nederlands
Polski
Tiếng Việt
Türkçe
简体中文
繁體中文
עִברִית
한국어
日本語










In [13]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION:
    The scraped text is from the career's page of a website.
    Your job is to extract the job postings and return them in JSON format containing the 
    following keys: `role`, `experience`, `skills` and `description`.
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE):
    """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data': page_data})
print(res.content)
type(res.content)

```json
{
  "role": "Retail Associate, PT - Nike Staten Island",
  "experience": "Part Time",
  "skills": [
    "Leading with service",
    "Understanding consumer needs",
    "Product knowledge",
    "Communication",
    "Teamwork",
    "Integrity"
  ],
  "description": "As a Nike Retail Associate, you bring the \"Just Do It\" mindset to life. You serve like a pro to help our consumers discover the product that inspires them - from living rooms to locker rooms - to move, dream, and dare. You’re in the store building brand trust and loyalty, but you don’t do it alone. We play in a fast-paced, high traffic environment, across store zones and consumer needs, but there’s no ego. We’re one team, showing up with optimism and hunger for the collective win."
}
```


str

In [14]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_res = json_parser.parse(res.content)

json_res

{'role': 'Retail Associate, PT - Nike Staten Island',
 'experience': 'Part Time',
 'skills': ['Leading with service',
  'Understanding consumer needs',
  'Product knowledge',
  'Communication',
  'Teamwork',
  'Integrity'],
 'description': 'As a Nike Retail Associate, you bring the "Just Do It" mindset to life. You serve like a pro to help our consumers discover the product that inspires them - from living rooms to locker rooms - to move, dream, and dare. You’re in the store building brand trust and loyalty, but you don’t do it alone. We play in a fast-paced, high traffic environment, across store zones and consumer needs, but there’s no ego. We’re one team, showing up with optimism and hunger for the collective win.'}

In [15]:
type(json_res)

dict

In [17]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")

df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [19]:
import chromadb, uuid

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links" : row["Links"]},
                       ids=[str(uuid.uuid4())])

In [22]:
links = collection.query(query_texts=job["skills"], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/kotlin-backend-portfolio'},
  {'links': 'https://example.com/typescript-frontend-portfolio'}],
 [{'links': 'https://example.com/android-tv-portfolio'},
  {'links': 'https://example.com/android-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/magento-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/android-tv-portfolio'}],
 [{'links': 'https://example.com/devops-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],
 [{'links': 'https://example.com/android-portfolio'},
  {'links': 'https://example.com/java-portfolio'}]]

In [23]:
job = json_res
job['skills']

['Leading with service',
 'Understanding consumer needs',
 'Product knowledge',
 'Communication',
 'Teamwork',
 'Integrity']